In [1]:
import numpy as np
import heapq
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks 
from scipy.interpolate import interp1d

## 1. WisePanda Neural Network Architecture

In [2]:
import numpy as np
import heapq
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

class VectorNet(nn.Module):
    def __init__(
        self,
    ) -> None:
        super(VectorNet, self).__init__()
        # 64 -> 55
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=5, kernel_size=10, padding="valid")
        self.a1 = nn.PReLU()
        # 55 -> 51
        self.conv2 = nn.Conv1d(in_channels=5, out_channels=5, kernel_size=5, padding="valid")
        self.a2 = nn.PReLU()
        self.fc1 = nn.Linear(5 * 51, 32)
        self.a3 = nn.PReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.a1(x)
        x = self.conv2(x)
        x = self.a2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.a3(x)
        return x


class CompareNet(nn.Module):
    def __init__(self):
        super(CompareNet, self).__init__()
        # 32 -> 23
        self.conv1 = nn.Conv1d(in_channels=2, out_channels=4, kernel_size=10, padding="valid")
        self.a1 = nn.PReLU()
        # 23 -> 19
        self.conv2 = nn.Conv1d(in_channels=4, out_channels=2, kernel_size=5, padding="valid")
        self.a2 = nn.PReLU()
        self.fc1 = nn.Linear(2 * 19, 1)
        self.a3 = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = self.a1(x)
        x = self.conv2(x)
        x = self.a2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.a3(x)
        return x

f_model = VectorNet()
c_model = CompareNet()

def inference(f_model, c_model, vector_1, vector_2):
    '''Description: This function is used to calculate the distance between two vectors'''
    f_model.eval()
    c_model.eval()
    feature_1 = f_model(vector_1)
    feature_2 = f_model(vector_2)
    dis = c_model(torch.stack((feature_1, feature_2), 1))
    return dis

## 2. Traditional Baseline Methods
The following section implements several traditional computational methods as baseline comparisons against WisePanda. These methods represent different approaches to vector similarity measurement and sequence alignment, serving as benchmarks to evaluate WisePanda's performance advantages.

In [3]:
# Drop-DTW (2021)
# from Drop-DTW: Aligning Common Signal Between Sequences While Dropping Outliers.
def drop_dtw(v1, v2, threshold=1.0):
    """
    Computes the Drop-DTW distance between two time series v1 and v2.

    Parameters:
    - v1 (list or np.array): The first time series.
    - v2 (list or np.array): The second time series.
    - threshold (float): The distance threshold for "dropping" a match.

    Returns:
    - float: The computed Drop-DTW distance.
    """
    n, m = len(v1), len(v2)
    dtw_matrix = np.zeros((n + 1, m + 1))

    # Initialize the distance matrix with infinity, except for the first element.
    dtw_matrix[1:, 0] = np.inf
    dtw_matrix[0, 1:] = np.inf
    dtw_matrix[0, 0] = 0

    penalty_factor=1.0  # Penalty factor for dropping

    # Fill the Drop-DTW matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(v1[i-1] - v2[j-1])
            
            # Drop-DTW's core logic: if the cost exceeds the threshold, "drop" the cost.
            if cost > threshold:
                # for costs exceeding the threshold, apply a penalty
                penalized_cost = (cost - threshold) * penalty_factor
                dtw_matrix[i, j] = penalized_cost + min(dtw_matrix[i-1, j],
                                                        dtw_matrix[i, j-1],
                                                        dtw_matrix[i-1, j-1])
            else:
                # for costs within the threshold, apply standard DTW logic
                dtw_matrix[i, j] = cost + min(dtw_matrix[i-1, j],
                                                dtw_matrix[i, j-1],
                                                dtw_matrix[i-1, j-1])


    # The Drop-DTW distance is the value at the bottom-right of the matrix.
    drop_dtw_distance = dtw_matrix[n, m]
    return drop_dtw_distance


In [4]:
# Soft-DTW (2017)
# from Soft-DTW: A Differentiable Loss Function for Time-Series
def soft_min(a, b, c, gamma):
    """
    Soft minimum with smoothing parameter gamma.
    When gamma -> 0, it approaches the true min.
    When gamma -> infinity, it approaches the average.
    """
    return -gamma * np.log(
        np.exp(-a/gamma) + np.exp(-b/gamma) + np.exp(-c/gamma)
    )

def soft_dtw(v1, v2, gamma=1.0):
    """
    Computes the Soft-DTW distance between two time series v1 and v2.

    Parameters:
    - v1 (list or np.array): The first time series.
    - v2 (list or np.array): The second time series.
    - gamma (float): The smoothing parameter for the soft-min function.
    
    Returns:
    - float: The computed Soft-DTW distance.
    """
    n, m = len(v1), len(v2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    # Fill the Soft-DTW matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(v1[i - 1] - v2[j - 1])
            
            # The core of Soft-DTW: using soft_min instead of traditional min
            dtw_matrix[i, j] = cost + soft_min(
                dtw_matrix[i - 1, j],    # Insertion
                dtw_matrix[i, j - 1],    # Deletion
                dtw_matrix[i - 1, j - 1],  # Match
                gamma
            )

    # The Soft-DTW distance is at the bottom-right of the matrix
    soft_dtw_distance = dtw_matrix[n, m]
    return soft_dtw_distance

In [5]:
# Event-DTW (2020)
# from EventDTW: An Improved Dynamic Time Warping Algorithm for Aligning Biomedical Signals of Nonuniform Sampling Frequencies
def event_dtw(v1, v2, window_size=3):
    '''
    Computes the Event-DTW distance between two time series.

    Parameters:
    - v1 (list or np.array): The first time series.
    - v2 (list or np.array): The second time series.
    
    Returns:
    - float: The computed Event-DTW distance.
    '''
    n, m = len(v1), len(v2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    # Define parameters for event detection and weighting
    #window_size = 3
    event_weight = 0.5  # < 1 to encourage event matching, > 1 to penalize it

    # Pre-compute event indices for v1 and v2
    def get_events(ts, window):
        events = set()
        for i in range(window, len(ts) - window):
            is_max = all(ts[i] >= ts[i - k] for k in range(1, window + 1)) and \
                     all(ts[i] >= ts[i + k] for k in range(1, window + 1))
            is_min = all(ts[i] <= ts[i - k] for k in range(1, window + 1)) and \
                     all(ts[i] <= ts[i + k] for k in range(1, window + 1))
            if is_max or is_min:
                events.add(i)
        return events
    
    events_v1 = get_events(v1, window_size)
    events_v2 = get_events(v2, window_size)

    # Fill the DTW matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(v1[i - 1] - v2[j - 1])
            
            # Apply different weights based on whether the points are events
            i_idx, j_idx = i - 1, j - 1
            
            weight = 1.0
            if i_idx in events_v1 and j_idx in events_v2:
                weight = event_weight            # Both are events - apply event_weight
            elif i_idx in events_v1 or j_idx in events_v2:
                weight = (1 + event_weight) / 2  # Only one is an event - apply a moderate weight
            # Otherwise, weight remains 1.0
            
            adjusted_cost = cost * weight
            
            dtw_matrix[i, j] = adjusted_cost + min(
                dtw_matrix[i - 1, j],    # Insertion
                dtw_matrix[i, j - 1],    # Deletion
                dtw_matrix[i - 1, j - 1] # Match
            )

    # The DTW distance is at the bottom-right of the matrix
    dtw_distance = dtw_matrix[n, m]
    return dtw_distance

In [6]:
# Modified COW (2022)
# from Modification of Correlation Optimized Warping Method for Position Alignment of Condition Measurements of Linear Assets
def modified_correlation_optimized_warping(v1, v2, segment_length=8, slack=3, boundary_flexibility=0.2):
    """
    Parameters:
    - v1, v2: Input sequences (lists of 64 height values)
    - segment_length: Length of each segment for warping
    - slack: Maximum allowed stretching/compression per segment
    - boundary_flexibility: Flexibility in start/end alignment (0-1)
    
    Returns:
    - float: Modified COW distance (negative correlation - lower is better)
    """
    if len(v1) != 64 or len(v2) != 64:
        raise ValueError("Both sequences must have exactly 64 elements")
    
    n = len(v1)
    num_segments = n // segment_length
    remainder = n % segment_length
    
    # Handle remainder by adjusting last segment
    if remainder > 0:
        num_segments += 1
    
    warped_v2 = []
    total_correlation = 0
    
    for seg in range(num_segments):
        # Calculate segment boundaries with flexibility
        if seg == 0:
            # First segment: allow flexible start
            flexible_start = int(boundary_flexibility * segment_length)
            start_idx = max(0, seg * segment_length - flexible_start)
            end_idx = (seg + 1) * segment_length
        elif seg == num_segments - 1:
            # Last segment: allow flexible end and handle remainder
            start_idx = seg * segment_length
            flexible_end = int(boundary_flexibility * segment_length)
            end_idx = min(n, start_idx + segment_length + flexible_end)
            if remainder > 0:
                end_idx = n  # Include all remaining points
        else:
            # Middle segments: standard boundaries
            start_idx = seg * segment_length
            end_idx = (seg + 1) * segment_length
        
        # Get reference segment
        ref_segment = np.array(v1[start_idx:end_idx])
        
        # Calculate adaptive slack based on position
        local_slack = slack
        if seg == 0 or seg == num_segments - 1:
            # Increase slack for boundary segments
            local_slack = min(slack + 1, segment_length // 2)
        
        # Extended target segment for warping search
        target_start = max(0, start_idx - local_slack)
        target_end = min(n, end_idx + local_slack)
        extended_target = np.array(v2[target_start:target_end])
        
        # Find best warping
        best_corr = -np.inf
        best_warped = None
        
        # Try different compression/expansion factors
        for compression in range(-local_slack, local_slack + 1):
            target_length = len(ref_segment) + compression
            
            if target_length < 2 or len(extended_target) < 2:
                continue
            
            try:
                # Create warped segment
                if target_length == len(extended_target):
                    warped_segment = extended_target
                else:
                    x_old = np.linspace(0, 1, len(extended_target))
                    x_new = np.linspace(0, 1, target_length)
                    f_interp = interpolate.interp1d(x_old, extended_target, kind='linear')
                    warped_segment = f_interp(x_new)
                
                # Adjust to match reference length
                if len(warped_segment) > len(ref_segment):
                    # Trim from center to preserve boundaries
                    trim_amount = len(warped_segment) - len(ref_segment)
                    start_trim = trim_amount // 2
                    warped_segment = warped_segment[start_trim:start_trim + len(ref_segment)]
                elif len(warped_segment) < len(ref_segment):
                    # Pad with edge values
                    pad_amount = len(ref_segment) - len(warped_segment)
                    pad_left = pad_amount // 2
                    pad_right = pad_amount - pad_left
                    warped_segment = np.concatenate([
                        np.full(pad_left, warped_segment[0]),
                        warped_segment,
                        np.full(pad_right, warped_segment[-1])
                    ])
                
                # Calculate correlation with distortion penalty
                if np.std(ref_segment) > 1e-8 and np.std(warped_segment) > 1e-8:
                    corr = np.corrcoef(ref_segment, warped_segment)[0, 1]
                    corr = float(corr)  # Ensure it's a Python float
                    
                    # Add robustness: penalize extreme distortions
                    distortion_penalty = abs(compression) / (local_slack + 1)
                    adjusted_corr = corr - 0.1 * distortion_penalty
                    
                    if not np.isnan(adjusted_corr) and adjusted_corr > best_corr:
                        best_corr = adjusted_corr
                        best_warped = warped_segment.copy()
                        
            except Exception:
                continue
        
        # Add warped segment to result
        if best_warped is not None:
            warped_v2.extend(best_warped)
            total_correlation += best_corr
        else:
            # Fallback: use original segment
            fallback_segment = v2[start_idx:end_idx]
            warped_v2.extend(fallback_segment)
            if np.std(ref_segment) > 1e-8 and np.std(fallback_segment) > 1e-8:
                corr = np.corrcoef(ref_segment, fallback_segment)[0, 1]
                total_correlation += float(corr)  # Ensure it's a Python float
    
    # Ensure exact output length
    if len(warped_v2) > 64:
        warped_v2 = warped_v2[:64]
    elif len(warped_v2) < 64:
        warped_v2.extend([warped_v2[-1]] * (64 - len(warped_v2)))
    
    # Return negative correlation as distance (lower = better alignment)
    weighted_correlation = total_correlation / num_segments
    return np.array(-weighted_correlation)

In [7]:
# Two-stage DTW（2024）
# from Two-stage dynamic time warping for intelligent fault detection of rotating machinery under variable speed
def _dtw_core(v1, v2, constraint_mask=None):
    """
    Core DTW computation with an optional constraint mask.
    """
    n, m = len(v1), len(v2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    if constraint_mask is None:
        constraint_mask = np.full((n + 1, m + 1), True)
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if constraint_mask[i, j]:
                cost = abs(v1[i-1] - v2[j-1])
                dtw_matrix[i, j] = cost + min(dtw_matrix[i-1, j],
                                              dtw_matrix[i, j-1],
                                              dtw_matrix[i-1, j-1])
    return dtw_matrix[n, m]

def two_stage_dtw(v1, v2, downsample_factor=4, window_size=5):
    """
    A conceptual two-stage DTW algorithm for robust time series matching.

    Parameters:
    - v1, v2 (np.array): The input time series.
    - downsample_factor (int): The factor for downsampling in the first stage.
    - window_size (int): The width of the constraint window for the second stage.
    
    Returns:
    - float: The final DTW distance.
    """
    n, m = len(v1), len(v2)

    # Stage 1: Coarse alignment on downsampled sequences
    v1_down = v1[::downsample_factor]
    v2_down = v2[::downsample_factor]
    
    n_down, m_down = len(v1_down), len(v2_down)
    
    dtw_matrix_down = np.full((n_down + 1, m_down + 1), np.inf)
    dtw_matrix_down[0, 0] = 0
    
    for i in range(1, n_down + 1):
        for j in range(1, m_down + 1):
            cost = abs(v1_down[i-1] - v2_down[j-1])
            dtw_matrix_down[i, j] = cost + min(dtw_matrix_down[i-1, j],
                                               dtw_matrix_down[i, j-1],
                                               dtw_matrix_down[i-1, j-1])

    # Backtrack to find the coarse path
    path = []
    i, j = n_down, m_down
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        min_val = min(dtw_matrix_down[i-1, j], dtw_matrix_down[i, j-1], dtw_matrix_down[i-1, j-1])
        if min_val == dtw_matrix_down[i-1, j-1]:
            i -= 1
            j -= 1
        elif min_val == dtw_matrix_down[i-1, j]:
            i -= 1
        else:
            j -= 1
    path.reverse()
    
    # Stage 2: Refined alignment with a constraint window
    constraint_mask = np.full((n + 1, m + 1), False)
    for i_down, j_down in path:
        i_orig = i_down * downsample_factor
        j_orig = j_down * downsample_factor
        
        for k in range(max(0, i_orig - window_size), min(n, i_orig + window_size + 1)):
            for l in range(max(0, j_orig - window_size), min(m, j_orig + window_size + 1)):
                constraint_mask[k+1, l+1] = True
    
    final_distance = _dtw_core(v1, v2, constraint_mask)
    return final_distance

The following methods are old appoaches that we used to compare with our model

In [8]:
# Fast Matching Method
def fast_matching_distance(v1, v2, distance_power=1):
    # Ensure the input vectors are numpy arrays
    v1 = np.asarray(v1)
    v2 = np.asarray(v2)
    
    # Check if the vectors have the same length
    assert v1.shape == v2.shape, "Vectors must be of the same shape"
    
    # Initialize the distance and known arrays
    distance = np.full(v1.shape, np.inf)
    known = np.zeros(v1.shape, dtype=bool)
    
    # Priority queue to store the distances
    heap = []

    # Initialize with the first element (or any specific start point)
    heapq.heappush(heap, (0, 0))
    distance[0] = 0
    known[0] = True

    # Define neighbors (for 1D, it's just the previous and next indices)
    neighbors = lambda idx: [idx - 1, idx + 1]

    while heap:
        dist, idx = heapq.heappop(heap)
        for n in neighbors(idx):
            if 0 <= n < len(v1) and not known[n]:
                new_dist = dist + abs(v1[n] - v2[n])**distance_power
                if new_dist < distance[n]:
                    distance[n] = new_dist
                    heapq.heappush(heap, (new_dist, n))
                    known[n] = True

    # The final score is the total accumulated distance
    total_distance = np.sum(distance)
    return total_distance

In [9]:
# Dynamic Time Warping (DTW)
def dynamic_time_warping(v1, v2, distance_power=1,):
    n, m = len(v1), len(v2)
    dtw_matrix = np.zeros((n+1, m+1))

    # Initialize the distance matrix with infinity
    dtw_matrix[1:, 0] = np.inf
    dtw_matrix[0, 1:] = np.inf

    # Set the first element to 0
    dtw_matrix[0, 0] = 0

    # Fill the DTW matrix
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = abs(v1[i-1] - v2[j-1])**distance_power
            # Take the minimum of the three possible transitions
            dtw_matrix[i, j] = cost + min(dtw_matrix[i-1, j],    # Insertion
                                          dtw_matrix[i, j-1],    # Deletion
                                          dtw_matrix[i-1, j-1])  # Match

    # The DTW distance is found at the bottom-right of the matrix
    dtw_distance = dtw_matrix[n, m]
    return dtw_distance

In [10]:
# Scale Invariant Signature (SIS)
def compute_curvature(v, velocity_power=1.5):
    """
    Calculate the curvature of a 1D curve.
    
    Parameters:
    - v: Input curve (1D array)
    - velocity_power: Power for velocity normalization (default=1.5)
                     Adjustable: 1.0 (less sensitive) to 2.0 (more sensitive)
    """
    dv = np.gradient(v)
    ddv = np.gradient(dv)
    
    # Add numerical stability
    denominator = np.maximum((1 + dv**2)**velocity_power, 1e-10)
    curvature = np.abs(ddv) / denominator
    
    # Handle NaN/Inf
    curvature = np.nan_to_num(curvature, nan=0.0, posinf=0.0, neginf=0.0)
    
    return curvature


def scale_invariant_signature(v, n_samples=None, velocity_power=1.5):
    """
    Generate a TRUE scale-invariant signature using curvature integral.
    
    KEY FIX: Use curvature integral as parameterization (not just normalized curvature!)
    When curve scales by λ: κ'=κ/λ, s'=λs → ∫κ'ds' = ∫κds (invariant!)
    
    Parameters:
    - v: Input curve (1D array)
    - n_samples: Output signature length (default=same as input)
    - velocity_power: Curvature parameter (default=1.5)
    
    Returns:
    - signature: Scale-invariant signature (1D array)
    """
    if n_samples is None:
        n_samples = len(v)
    
    # Step 1: Compute curvature
    curvature = compute_curvature(v, velocity_power=velocity_power)
    
    # Step 2: Compute arc length parameterization
    dv = np.gradient(v)
    arc_length_element = np.sqrt(1 + dv**2)
    arc_length = np.cumsum(arc_length_element)
    arc_length = np.insert(arc_length, 0, 0)  # Add starting point
    
    # Step 3: KEY - Compute curvature integral (this is scale-invariant!)
    # θ(s) = ∫₀ˢ κ(σ)dσ
    curvature_integral = np.cumsum(curvature * arc_length_element)
    curvature_integral = np.insert(curvature_integral, 0, 0)
    
    # Step 4: Normalize to [0, 1]
    if curvature_integral[-1] > 1e-10:
        param = curvature_integral / curvature_integral[-1]
    else:
        # Straight line case
        param = np.linspace(0, 1, len(curvature_integral))
    
    if arc_length[-1] > 1e-10:
        arc_normalized = arc_length / arc_length[-1]
    else:
        arc_normalized = np.linspace(0, 1, len(arc_length))
    
    # Step 5: Signature = normalized arc length as function of curvature integral
    # Resample to n_samples points
    f = interp1d(param, arc_normalized, kind='linear', 
                 bounds_error=False, fill_value='extrapolate')
    
    uniform_param = np.linspace(0, 1, n_samples)
    signature = f(uniform_param)
    
    return signature


def scale_invariant_signatures(sig1, sig2, distance_metric='l2'):
    """
    Compare two scale-invariant signatures.
    
    Parameters:
    - sig1, sig2: Two signatures (must have same length)
    - distance_metric: 'l2' (default), 'l1', or 'correlation'
    
    Returns:
    - distance: Distance between signatures
    """
    # Ensure same length (interpolate if needed)
    if len(sig1) != len(sig2):
        n = max(len(sig1), len(sig2))
        f1 = interp1d(np.linspace(0, 1, len(sig1)), sig1, kind='linear')
        f2 = interp1d(np.linspace(0, 1, len(sig2)), sig2, kind='linear')
        x = np.linspace(0, 1, n)
        sig1, sig2 = f1(x), f2(x)
    
    # Compute distance
    if distance_metric == 'l2':
        return np.sqrt(np.sum((sig1 - sig2)**2))
    elif distance_metric == 'l1':
        return np.sum(np.abs(sig1 - sig2))
    elif distance_metric == 'correlation':
        l2_dist = np.sqrt(np.sum((sig1 - sig2)**2))
        l1_dist = np.sum(np.abs(sig1 - sig2))
        return l2_dist + l1_dist/len(sig1)

## 3. Dataset Loading and Validation Setup
This section loads manually curated datasets that provide ground truth pairings for algorithm performance validation. The datasets consist of expertly matched ancient document fragments with known correspondences.

Core Datasets

&nbsp;&nbsp;&nbsp;&nbsp;**vector_real_118_patch.npy**: Bamboo slip dataset (Bamboo236) containing 118 pairs of matched bamboo fragments.<br>
&nbsp;&nbsp;&nbsp;&nbsp;**vector_real_335_patch.npy**: Wooden slip dataset (Wood670) containing 335 pairs of matched wooden document fragments.

Extended Datasets (Interference Data)

&nbsp;&nbsp;&nbsp;&nbsp;**interference_data_bamboo.npy**, **interference_data_bamboo_bottom.npy**, **interference_data_wood.npy** and **interference_data_wood_bottom.npy**: Additional unmatched fragments used to expand the candidate pool.


In [14]:
def get_top_k_accuracy(k, dis_map, direction):
    '''Description: This function is used to calculate the top k accuracy of the model'''
    # calculate the top k accuracy of the model
    top_k = 0
    length = len(dis_map)
    length = 335
    gather_list_transverse = []
    for i in range(length):
        dis_list = []
        if direction == "longitudinal":
           dis_list = [x[i] for x in dis_map]
        else: 
            dis_list = dis_map[i].copy()
        dis_list.sort()

        top_index = dis_list.index(dis_map[i][i])
        gather_list_transverse.append(top_index+1)
    for j in gather_list_transverse:
        if j <= k:
            top_k += 1
     
    return top_k

def calculate_position_stats_numpy(list1, list2, list3):
    # transform to numpy array
    arrays = np.array([list1, list2, list3])
    
    # calculate mean and standard deviation
    means = np.mean(arrays, axis=0)
    stds = np.std(arrays, axis=0, ddof=1)  

    means = np.round(means, 2)
    stds = np.round(stds, 2)
    
    return means.tolist(), stds.tolist()

In [15]:
def get_topk_accuracy_with_extend(extend_number, method, parameter=None, f_model_pth=None, c_model_pth=None):
    # data loading
    # real_vectors = np.load("dataset/vector_real_118_patch.npy")
    # real_vectors = real_vectors[:, 2:4, :].astype(np.float32)
    # interference_data = np.load("dataset/interference_data_bamboo.npy")
    # interference_data = interference_data[:extend_number]

    real_vectors = np.load("dataset/vector_real_335_patch.npy")
    real_vectors = real_vectors[:, 2:4, :].astype(np.float32)
    interference_data = np.load("dataset/interference_data_wood.npy")
    interference_data= interference_data[:extend_number]

    
    # data connection
    top_origin_list = []
    bottom_list = []
    for i in real_vectors:
        top_origin_list.append(i[0])
        bottom_list.append(i[1])
    for i in interference_data:
        top_origin_list.append(i)
        bottom_list.append(i)

    # model loading
    f_model_pth = f_model_pth if f_model_pth else 'models/f_model_1.pth'
    c_model_pth = c_model_pth if c_model_pth else 'models/c_model_1.pth'

    f_model.load_state_dict(torch.load(f_model_pth))
    c_model.load_state_dict(torch.load(c_model_pth))    

    length = len(top_origin_list)
    dis_map = []
    for i in range(length):
        dis_list = []
        for j in range(length):
            if method == "wisepanda" or method == "GAN" or method == "seriesGAN" or method == "Diffusion" or method == "Diffusion-TS":
                # wisepanda
                v1 = top_origin_list[i].reshape(1, 1, -1)
                v2 = bottom_list[j].reshape(1, 1, -1)

                v1 = v1 - v1.min()  
                v2 = v2 - v2.min()  

                v1 = v1 / (v1.max() + 1e-6) 
                v2 = v2 / (v2.max() + 1e-6)

                v1 = torch.tensor(v1, dtype=torch.float32)
                v2 = torch.tensor(v2, dtype=torch.float32)

                dis = inference(f_model, c_model, v1, v2)
                dis_list.append(dis.item())
            else:   
                # compare
                v1 = top_origin_list[i][np.newaxis, np.newaxis, :]
                v2 = bottom_list[j][np.newaxis, np.newaxis, :]
                if method == "drop_dtw":
                    dis = drop_dtw(v1[0][0], v2[0][0], threshold=parameter if parameter else 1.0)
                elif method == "event_dtw":
                     dis = event_dtw(v1[0][0], v2[0][0], window_size=parameter if parameter else 3)
                elif method == "two_stage_dtw":
                     dis = two_stage_dtw(v1[0][0], v2[0][0], window_size=parameter if parameter else 5)
                elif method == "modified_cow":
                     dis = modified_correlation_optimized_warping(v1[0][0], v2[0][0], segment_length=parameter if parameter else 8, slack=3, boundary_flexibility=0.2)
                elif method == "fast_matching":
                     dis = fast_matching_distance(v1[0][0], v2[0][0], distance_power=parameter if parameter else 1)
                elif method == "dtw":
                     dis = dynamic_time_warping(v1[0][0], v2[0][0], distance_power=parameter if parameter else 1)
                elif method == "sis":
                     dis = scale_invariant_signatures(v1[0][0], v2[0][0], distance_metric= parameter if parameter else 'l2')  

                dis_list.append(dis.item())
        dis_map.append(dis_list)
    k_list = [1, 5, 10, 20, 50, 100]
    result_list = []
    for i in k_list:
        k = get_top_k_accuracy(i, dis_map, "longitudinal")
        k_ = get_top_k_accuracy(i, dis_map, "transverse")
        result_list.append(round((k+k_)/670, 4))

    return dis_map, result_list

In [ ]:
# drop_dtw (2021)
parameter_list = [0.1, 0.5, 1.0]
means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="drop_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("drop_dtw (2021) (wood670) & means:", means)
print("drop_dtw (2021) (wood670) & stds:", stds)

In [57]:
# event_dtw (2020)
parameter_list = [1, 3, 5]
means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="event_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("event_dtw (2020) (wood670) & means:", means)
print("event_dtw (2020) (wood670) & stds:", stds)

result in original data (wood670):
event_dtw (2020) (wood670) & means: [2.84, 7.06, 10.95, 15.87, 23.38, 28.81]
event_dtw (2020) (wood670) & stds: [0.25, 0.52, 0.08, 0.62, 0.23, 0.26]


In [55]:
# modified_cow (2022）
parameter_list = [8, 10, 12]
means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="modified_cow", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("modified_cow (2022) (wood670) & means:", means)
print("modified_cow (2022) (wood670) & stds:", stds)

result in original data (wood670):
modified_cow (2022) (wood670) & means: [0.95, 2.89, 5.02, 7.86, 14.53, 22.59]
modified_cow (2022) (wood670) & stds: [0.22, 0.45, 0.62, 1.12, 1.45, 1.25]


In [19]:
# two_stage_dtw (2025)
parameter_list = [3, 5, 7]
means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="two_stage_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("two_stage_dtw (2025) (wood670) & means:", means)
print("two_stage_dtw (2025) (wood670) & stds:", stds)

result in original data (wood670):
two_stage_dtw (2025) (wood670) & means: [8.21, 22.79, 34.38, 49.85, 71.74, 84.73]
two_stage_dtw (2025) (wood670) & stds: [0.4, 1.1, 0.09, 0.15, 0.23, 0.23]


In [21]:
# two_stage_dtw (2025)
parameter_list = ['l1', 'l2', 'correlation']
means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="sis", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("sis (wood670) & means:", means)
print("sis (wood670) & stds:", stds)

result in original data (wood670):
sis (wood670) & means: [9.95, 26.96, 36.97, 49.15, 69.95, 82.69]
sis (wood670) & stds: [0.09, 0.6, 0.69, 0.09, 0.38, 0.4]


In [18]:
# GAN
valid_info = ['normal', 'GAN']
means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results   
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("GAN in normal (wood670) & means:", means)
print("GAN in normal (wood670) & stds:", stds)

result in original data (wood670):
GAN in normal (wood670) & means: [1.39, 6.96, 13.04, 23.03, 48.46, 75.03]
GAN in normal (wood670) & stds: [0.17, 0.22, 0.48, 1.2, 2.74, 2.43]


In [52]:
# seriesGAN
valid_info = ['normal', 'seriesGAN']
means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("seriesGAN in normal (wood670) & means:", means)
print("seriesGAN in normal (wood670) & stds:", stds)

result in original data (wood670):
seriesGAN in normal (wood670) & means: [0.6, 2.94, 5.07, 8.41, 17.21, 26.97]
seriesGAN in normal (wood670) & stds: [0.15, 0.37, 0.91, 1.25, 0.75, 0.56]


In [53]:
# Diffusion
valid_info = ['normal', 'Diffusion']
means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("Diffusion in normal (wood670) & means:", means)
print("Diffusion in normal (wood670) & stds:", stds)

result in original data (wood670):
Diffusion in normal (wood670) & means: [0.4, 1.29, 2.59, 4.73, 10.45, 18.16]
Diffusion in normal (wood670) & stds: [0.09, 0.23, 0.43, 0.7, 1.42, 0.99]


In [54]:
# Diffusion-TS
valid_info = ['normal', 'Diffusion-TS']
means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("Diffusion-TS in normal (wood670) & means:", means)
print("Diffusion-TS in normal (wood670) & stds:", stds)

result in original data (wood670):
Diffusion-TS in normal (wood670) & means: [0.55, 2.14, 4.33, 7.86, 15.92, 25.17]
Diffusion-TS in normal (wood670) & stds: [0.23, 0.61, 0.65, 1.13, 1.04, 1.53]


In [17]:
# wisepanda
valid_info = ['normal', 'wisepanda']
means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
print("result in original data (wood670):")
print("Wisepanda in normal (wood670) & means:", means)
print("Wisepanda in normal (wood670) & stds:", stds)

result in original data (wood670):
Wisepanda in normal (wood670) & means: [5.27, 16.47, 26.67, 39.45, 63.08, 86.07]
Wisepanda in normal (wood670) & stds: [0.31, 0.96, 0.85, 1.12, 1.65, 0.23]


## 4. Performance Evaluation Results
**Performance Testing (No Extended Data)**

The following section evaluates each method's performance under baseline conditions without interference data (extend_number = 0). Results are reported for Top-k accuracy where k = [1, 5, 10, 20, 50, 100], representing the accuracy when the correct match appears within the top k candidates.

In [38]:
# drop_dtw (2021)
parameter_list = [0.1, 0.5, 1.0]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="drop_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="drop_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("drop_dtw (2021) (236) & means:", means)
print("drop_dtw (2021) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("drop_dtw (2021) (1350) & means:", extend_means)
print("drop_dtw (2021) (1350) & stds:", extend_stds)

result in original data (bamboo236):
drop_dtw (2021) (236) & means: [8.34, 23.02, 31.78, 49.29, 75.56, 94.92]
drop_dtw (2021) (236) & stds: [1.22, 0.65, 0.42, 0.25, 0.65, 0.0]
result in extended data (bamboo1114):
drop_dtw (2021) (1350) & means: [4.24, 11.16, 14.41, 19.92, 29.66, 37.29]
drop_dtw (2021) (1350) & stds: [0.43, 0.24, 0.43, 0.0, 0.0, 0.74]


In [39]:
# event_dtw (2020)
parameter_list = [1, 3, 5]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="event_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="event_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("event_dtw (2020) (236) & means:", means)
print("event_dtw (2020) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("event_dtw (2020) (1350) & means:", extend_means)
print("event_dtw (2020) (1350) & stds:", extend_stds)

result in original data (bamboo236):
event_dtw (2020) (236) & means: [7.63, 24.01, 31.35, 48.45, 75.42, 94.63]
event_dtw (2020) (236) & stds: [0.0, 0.65, 0.73, 1.07, 1.12, 0.25]
result in extended data (bamboo1114):
event_dtw (2020) (1350) & means: [4.09, 12.85, 15.96, 22.46, 34.04, 42.94]
event_dtw (2020) (1350) & stds: [0.49, 0.65, 1.07, 1.47, 0.88, 2.0]


In [40]:
# modified_cow (2022）
parameter_list = [8, 10, 12]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="modified_cow", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="modified_cow", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("modified_cow (2022) (236) & means:", means)
print("modified_cow (2022) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("modified_cow (2022) (1350) & means:", extend_means)
print("modified_cow (2022) (1350) & stds:", extend_stds)

result in original data (bamboo236):
modified_cow (2022) (236) & means: [8.05, 24.44, 38.42, 57.06, 85.03, 98.59]
modified_cow (2022) (236) & stds: [0.85, 0.25, 1.71, 1.22, 0.65, 0.65]
result in extended data (bamboo1114):
modified_cow (2022) (1350) & means: [2.97, 9.18, 13.7, 19.35, 28.96, 41.53]
modified_cow (2022) (1350) & stds: [0.85, 0.65, 1.48, 1.71, 1.71, 2.55]


In [41]:
# two_stage_dtw (2025)
parameter_list = [3, 5, 7]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="two_stage_dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="two_stage_dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("two_stage_dtw (2025) (236) & means:", means)
print("two_stage_dtw (2025) (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("two_stage_dtw (2025) (1350) & means:", extend_means)
print("two_stage_dtw (2025) (1350) & stds:", extend_stds)

result in original data (bamboo236):
two_stage_dtw (2025) (236) & means: [7.63, 23.45, 30.65, 49.44, 74.58, 95.34]
two_stage_dtw (2025) (236) & stds: [0.0, 0.49, 0.24, 0.25, 0.0, 0.0]
result in extended data (bamboo1114):
two_stage_dtw (2025) (1350) & means: [4.1, 12.29, 14.41, 19.78, 29.38, 36.86]
two_stage_dtw (2025) (1350) & stds: [0.25, 0.0, 0.0, 0.65, 0.24, 0.43]


In [42]:
# Fast Matching Method
parameter_list = [0.8, 1, 1.2]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="fast_matching", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="fast_matching", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("fast_matching (236) & means:", means)
print("fast_matching (236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("fast_matching (1350) & means:", extend_means)
print("fast_matching (1350) & stds:", extend_stds)

result in original data (bamboo236):
fast_matching (236) & means: [8.9, 18.64, 26.41, 37.71, 65.25, 94.35]
fast_matching (236) & stds: [0.85, 0.0, 0.65, 1.53, 1.7, 0.24]
result in extended data (bamboo1114):
fast_matching (1350) & means: [5.79, 11.02, 12.85, 15.4, 23.45, 32.2]
fast_matching (1350) & stds: [1.36, 0.0, 0.25, 0.49, 1.49, 1.53]


In [43]:
# DTW
parameter_list = [0.8, 1, 1.2]
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="dtw", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="dtw", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("dynamic_time_warping(236) & means:", means)
print("dynamic_time_warping(236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("dynamic_time_warping (1350) & means:", extend_means)
print("dynamic_time_warping (1350) & stds:", extend_stds)

result in original data (bamboo236):
dynamic_time_warping(236) & means: [8.05, 22.88, 31.36, 48.73, 74.72, 94.63]
dynamic_time_warping(236) & stds: [0.42, 0.43, 0.43, 1.47, 2.0, 0.89]
result in extended data (bamboo1114):
dynamic_time_warping (1350) & means: [3.95, 11.72, 14.12, 19.78, 29.8, 37.29]
dynamic_time_warping (1350) & stds: [0.25, 0.24, 0.25, 0.65, 1.91, 1.12]


In [44]:
# Scale Invariant Signature (SIS)
parameter_list = ['l1', 'l2', 'correlation']
means_stds_list = []
extend_means_stds_list = []
for i in parameter_list:
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="sis", parameter=i)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="sis", parameter=i)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("scale_invariant_signatures(236) & means:", means)
print("scale_invariant_signatures(236) & stds:", stds)
print("result in extended data (bamboo1114):")
print("scale_invariant_signatures (1350) & means:", extend_means)
print("scale_invariant_signatures (1350) & stds:", extend_stds)

result in original data (bamboo236):
scale_invariant_signatures(236) & means: [10.45, 21.47, 29.38, 44.07, 71.47, 94.07]
scale_invariant_signatures(236) & stds: [0.24, 0.98, 2.34, 4.43, 2.45, 0.0]
result in extended data (bamboo1114):
scale_invariant_signatures (1350) & means: [7.34, 14.12, 17.94, 22.04, 32.35, 44.21]
scale_invariant_signatures (1350) & stds: [0.25, 1.29, 1.61, 1.85, 3.45, 4.53]


## 5. Physic model results

In [ ]:
# wisepanda
valid_info = ['normal', 'wisepanda']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1000, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("wisepanda in normal (236) & means:", means)
print("wisepanda in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("wisepanda in normal (1350) & means:", extend_means)
print("wisepanda in normal (1350) & stds:", extend_stds)

## 6. Generated model results

### normal

In [133]:
# GAN
valid_info = ['normal', 'GAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results   
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("GAN in normal (236) & means:", means)
print("GAN in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("GAN in normal (1350) & means:", extend_means)
print("GAN in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
GAN in normal (236) & means: [4.52, 17.94, 32.06, 54.52, 81.36, 96.61]
GAN in normal (236) & stds: [1.07, 1.07, 3.66, 2.45, 0.85, 0.73]
result in extend data (bamboo1350):
GAN in normal (1350) & means: [0.42, 5.51, 8.47, 13.84, 29.38, 42.51]
GAN in normal (1350) & stds: [0.73, 1.53, 2.24, 1.71, 2.48, 2.45]


In [ ]:
# seriesGAN
valid_info = ['normal', 'seriesGAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("seriesGAN in normal (236) & means:", means)
print("seriesGAN in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("seriesGAN in normal (1350) & means:", extend_means)
print("seriesGAN in normal (1350) & stds:", extend_stds)

In [135]:
# Diffusion
valid_info = ['normal', 'Diffusion']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion in normal (236) & means:", means)
print("Diffusion in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion in normal (1350) & means:", extend_means)
print("Diffusion in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion in normal (236) & means: [3.39, 10.03, 19.63, 34.75, 70.48, 96.05]
Diffusion in normal (236) & stds: [0.42, 0.88, 2.55, 0.85, 5.52, 0.98]
result in extend data (bamboo1350):
Diffusion in normal (1350) & means: [1.27, 4.66, 8.47, 12.85, 27.82, 43.22]
Diffusion in normal (1350) & stds: [0.42, 1.27, 2.58, 2.17, 0.49, 3.2]


In [136]:
# Diffusion-TS
valid_info = ['normal', 'Diffusion-TS']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion-TS in normal (236) & means:", means)
print("Diffusion-TS in normal (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion-TS in normal (1350) & means:", extend_means)
print("Diffusion-TS in normal (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion-TS in normal (236) & means: [3.39, 14.83, 27.26, 45.91, 80.37, 98.73]
Diffusion-TS in normal (236) & stds: [1.47, 1.27, 2.82, 2.82, 3.55, 0.85]
result in extend data (bamboo1350):
Diffusion-TS in normal (1350) & means: [0.7, 3.11, 4.94, 10.17, 20.48, 31.64]
Diffusion-TS in normal (1350) & stds: [0.49, 2.33, 3.23, 5.61, 9.14, 8.95]


### low-corrosion

In [137]:
# GAN
valid_info = ['low-corrosion', 'GAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("GAN in low-corrosion (236) & means:", means)
print("GAN in low-corrosion (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("GAN in low-corrosion (1350) & means:", extend_means)
print("GAN in low-corrosion (1350) & stds:", extend_stds)

result in original data (bamboo236):
GAN in low-corrosion (236) & means: [6.64, 21.75, 36.86, 55.51, 83.19, 99.72]
GAN in low-corrosion (236) & stds: [1.22, 3.8, 4.04, 1.94, 0.48, 0.24]
result in extend data (bamboo1350):
GAN in low-corrosion (1350) & means: [2.68, 7.34, 10.73, 16.24, 30.65, 42.23]
GAN in low-corrosion (1350) & stds: [0.98, 2.01, 1.22, 4.49, 0.88, 1.71]


In [138]:
# seriesGAN
valid_info = ['low-corrosion', 'seriesGAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("seriesGAN in low-corrosion (236) & means:", means)
print("seriesGAN in low-corrosion (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("seriesGAN in low-corrosion (1350) & means:", extend_means)
print("seriesGAN in low-corrosion (1350) & stds:", extend_stds)

result in original data (bamboo236):
seriesGAN in low-corrosion (236) & means: [10.73, 27.54, 42.37, 58.47, 84.75, 97.88]
seriesGAN in low-corrosion (236) & stds: [1.91, 2.58, 1.12, 0.73, 2.94, 1.85]
result in extend data (bamboo1350):
seriesGAN in low-corrosion (1350) & means: [3.39, 10.31, 17.38, 25.71, 39.12, 50.14]
seriesGAN in low-corrosion (1350) & stds: [1.53, 2.17, 1.47, 5.69, 5.4, 5.02]


In [139]:
# Diffusion
valid_info = ['low-corrosion', 'Diffusion']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion in low-corrosion (236) & means:", means)
print("Diffusion in low-corrosion (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion in low-corrosion (1350) & means:", extend_means)
print("Diffusion in low-corrosion (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion in low-corrosion (236) & means: [7.2, 19.21, 32.63, 52.97, 85.45, 99.29]
Diffusion in low-corrosion (236) & stds: [1.95, 2.13, 2.55, 4.77, 0.88, 0.65]
result in extend data (bamboo1350):
Diffusion in low-corrosion (1350) & means: [2.26, 8.9, 13.14, 20.34, 31.35, 44.21]
Diffusion in low-corrosion (1350) & stds: [1.71, 4.48, 3.31, 5.54, 7.53, 7.31]


In [140]:
# Diffusion-TS
valid_info = ['low-corrosion', 'Diffusion-TS']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion-TS in low-corrosion (236) & means:", means)
print("Diffusion-TS in low-corrosion (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion-TS in low-corrosion (1350) & means:", extend_means)
print("Diffusion-TS in low-corrosion (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion-TS in low-corrosion (236) & means: [4.8, 21.89, 33.76, 53.53, 84.32, 99.01]
Diffusion-TS in low-corrosion (236) & stds: [0.24, 2.34, 4.57, 2.49, 1.53, 0.88]
result in extend data (bamboo1350):
Diffusion-TS in low-corrosion (1350) & means: [1.69, 7.49, 11.58, 19.07, 31.21, 42.37]
Diffusion-TS in low-corrosion (1350) & stds: [0.73, 1.6, 1.29, 1.85, 1.71, 1.95]


### perturbation

In [141]:
# GAN
valid_info = ['perturbation', 'GAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("GAN in perturbation (236) & means:", means)
print("GAN in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("GAN in perturbation (1350) & means:", extend_means)
print("GAN in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
GAN in perturbation (236) & means: [4.1, 15.4, 27.68, 49.72, 81.64, 96.89]
GAN in perturbation (236) & stds: [0.65, 1.07, 2.89, 4.27, 1.29, 0.24]
result in extend data (bamboo1350):
GAN in perturbation (1350) & means: [1.41, 7.2, 12.71, 19.91, 37.15, 54.66]
GAN in perturbation (1350) & stds: [0.65, 1.95, 2.12, 2.58, 3.29, 2.12]


In [142]:
# seriesGAN
valid_info = ['perturbation', 'seriesGAN']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("seriesGAN in perturbation (236) & means:", means)
print("seriesGAN in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("seriesGAN in perturbation (1350) & means:", extend_means)
print("seriesGAN in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
seriesGAN in perturbation (236) & means: [3.81, 21.89, 36.44, 56.21, 78.96, 96.47]
seriesGAN in perturbation (236) & stds: [0.43, 1.3, 1.27, 4.69, 1.71, 0.65]
result in extend data (bamboo1350):
seriesGAN in perturbation (1350) & means: [0.99, 5.79, 11.87, 21.75, 42.09, 57.63]
seriesGAN in perturbation (1350) & stds: [0.65, 1.77, 2.78, 5.46, 6.81, 5.51]


In [143]:
# Diffusion
valid_info = ['perturbation', 'Diffusion']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion in perturbation (236) & means:", means)
print("Diffusion in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion in perturbation (1350) & means:", extend_means)
print("Diffusion in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion in perturbation (236) & means: [6.5, 20.62, 33.05, 50.57, 79.8, 97.74]
Diffusion in perturbation (236) & stds: [1.3, 3.94, 3.47, 5.72, 1.29, 0.88]
result in extend data (bamboo1350):
Diffusion in perturbation (1350) & means: [1.55, 6.07, 10.03, 17.38, 33.62, 47.32]
Diffusion in perturbation (1350) & stds: [0.88, 2.09, 2.13, 2.24, 2.98, 4.67]


In [144]:
# Diffusion-TS
valid_info = ['perturbation', 'Diffusion-TS']
means_stds_list = []
extend_means_stds_list = []
for i in range(3):
    f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(i+1)+'.pth'
    c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(i+1)+'.pth'
    dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    extend_dis_map, extend_result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)
    # append the results
    result_list = [x * 100 for x in result_list]
    means_stds_list.append(result_list)
    # append the extend results (1350)
    extend_result_list = [x * 100 for x in extend_result_list]
    extend_means_stds_list.append(extend_result_list)
means, stds = calculate_position_stats_numpy(means_stds_list[0], means_stds_list[1], means_stds_list[2])
extend_means, extend_stds = calculate_position_stats_numpy(extend_means_stds_list[0], extend_means_stds_list[1], extend_means_stds_list[2])
print("result in original data (bamboo236):")
print("Diffusion-TS in perturbation (236) & means:", means)
print("Diffusion-TS in perturbation (236) & stds:", stds)
print("result in extend data (bamboo1350):")
print("Diffusion-TS in perturbation (1350) & means:", extend_means)
print("Diffusion-TS in perturbation (1350) & stds:", extend_stds)

result in original data (bamboo236):
Diffusion-TS in perturbation (236) & means: [2.68, 11.16, 22.32, 41.25, 77.54, 96.61]
Diffusion-TS in perturbation (236) & stds: [1.49, 2.18, 3.12, 3.43, 1.85, 0.42]
result in extend data (bamboo1350):
Diffusion-TS in perturbation (1350) & means: [0.57, 2.68, 3.95, 9.61, 21.75, 39.41]
Diffusion-TS in perturbation (1350) & stds: [0.49, 1.37, 1.49, 2.76, 5.36, 6.58]


## 7. Permutation tests

In [ ]:
def get_top_k_list(dis_map, direction):
    '''Description: This function is used to calculate the top k accuracy of the model'''
    # calculate the top k accuracy of the model
    length = len(dis_map)
    length = 118
    gather_list_transverse = []
    for i in range(length):
        dis_list = []
        if direction == "longitudinal":
           dis_list = [x[i] for x in dis_map]
        else: 
            dis_list = dis_map[i].copy()
        dis_list.sort()

        top_index = dis_list.index(dis_map[i][i])
        gather_list_transverse.append(top_index+1)
    
    return gather_list_transverse

def get_diagonal_ranks(dis_map):
    """Get the ranks of the diagonal elements in the distance map."""
    ranks = []
    longtitude_ranks = get_top_k_list(dis_map=dis_map, direction="longitudinal")
    Latitude_ranks = get_top_k_list(dis_map=dis_map, direction="latitude")
    ranks = longtitude_ranks + Latitude_ranks
    return np.array(ranks)

def calculate_accuracy(ranks, top_n=50):
    """calculate the top-n accuracy"""
    ranks_array = np.array(ranks) 
    comparison_result = ranks_array <= top_n

    return np.mean(comparison_result)

def permutation_test(wise_panda_ranks, sis_ranks, top_n=50, n_permutations=1000):
    """
    Perform a permutation test to compare the accuracies of WisePanda and SIS.
    Args:
        wise_panda_ranks (array-like): Ranks from WisePanda method.
        sis_ranks (array-like): Ranks from SIS method.
        top_n (int): The threshold for top-n accuracy.
        n_permutations (int): Number of permutations to perform.
    Returns:
        observed_diff (float): Observed difference in accuracies.
        permutation_diffs (list): List of differences from each permutation.
        p_value (float): p-value from the permutation test.
    """
    # calculate observed difference
    sis_accuracy = calculate_accuracy(sis_ranks, top_n)
    wp_accuracy = calculate_accuracy(wise_panda_ranks, top_n)
    
    observed_diff = wp_accuracy - sis_accuracy
    
    # combine the data
    all_ranks = np.concatenate([wise_panda_ranks, sis_ranks])
    n = len(wise_panda_ranks)  # size of one group
    
    # perform permutations
    permutation_diffs = []
    
    for _ in range(n_permutations):
        # shuffle the combined data
        np.random.shuffle(all_ranks)
        
        # split into two new groups
        perm_wp = all_ranks[:n]
        perm_sis = all_ranks[n:]
        
        # calculate the difference in accuracies
        perm_wp_acc = calculate_accuracy(perm_wp, top_n)
        perm_sis_acc = calculate_accuracy(perm_sis, top_n)
        perm_diff = perm_wp_acc - perm_sis_acc
        
        permutation_diffs.append(perm_diff)
    
    # calculate p-value
    p_value = np.mean(np.abs(permutation_diffs) >= np.abs(observed_diff))
    
    return observed_diff, permutation_diffs, p_value

def plot_permutation_results(observed_diff, permutation_diffs):
    """Plot histogram of permutation test results"""
    plt.figure(figsize=(10, 6))
    plt.hist(permutation_diffs, bins=30, alpha=0.7, label='Distribution of permuted differences')
    plt.axvline(observed_diff, color='red', linestyle='--', 
                label=f'Observed difference: {observed_diff:.4f}')
    plt.xlabel('Accuracy difference')
    plt.ylabel('Frequency')
    plt.title('Permutation Test Results')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def evaluate_multiple_topk(rank_matrix1, rank_matrix2, top_k_list=[1, 5, 10, 20, 50, 100], n_permutations=1000):
    """
    Evaluate permutation test for multiple top-k values
    """
    results = []
    
    for top_n in top_k_list:
        observed_diff, permutation_diffs, p_value = permutation_test(
            rank_matrix1, rank_matrix2, top_n=top_n, n_permutations=n_permutations
        )
        
        results.append({
            'Top-k': f'Top-{top_n}',
            'Observed Difference': f'{observed_diff:.4f}',
            'P-value': f'{p_value:.4f}',
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })

    results_df = pd.DataFrame(results)
    # Transpose and reset index
    results_df = results_df.set_index('Top-k').T.reset_index()
    results_df.rename(columns={'index': 'Top-k'}, inplace=True)
    return results_df

def run_permutation_test(current_dis_map, wisepanda_dis_map):
    # wp_dis_map_array = np.array(wisepanda_dis_map)
    # dis_map_array = np.array(current_dis_map)

    wp_rank_matrix = get_diagonal_ranks(wisepanda_dis_map)
    ddtw_rank_matrix = get_diagonal_ranks(current_dis_map)

    results = evaluate_multiple_topk(
        wp_rank_matrix, 
        ddtw_rank_matrix, 
        top_k_list=[1, 5, 10, 20, 50, 100],
        n_permutations=1000
    )
    return results.to_string(index=False)

In [399]:
valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

### 7.1 Curvematching methods

#### 7.1.1 Drop-DTW vs WisePanda

In [365]:
# Drop-DTW (2021)
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="drop_dtw", parameter=0.1)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Drop-DTW vs WisePanda:")
print(table)

Drop-DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0466 0.1271 0.2119 0.1992 0.1864  0.0508
            P-value 0.1230 0.0020 0.0000 0.0000 0.0000  0.0000
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [366]:
# Drop-DTW (2021)
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="drop_dtw", parameter=0.1)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Drop-DTW vs WisePanda:")
print(table)

Drop-DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0805 0.2500 0.3814 0.5000 0.6229  0.5212
            P-value 0.0040 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.2 Event-DTW vs WisePanda

In [367]:
# Event-DTW (2020)
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="event_dtw", parameter=1)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Event-DTW vs WisePanda:")
print(table)

Event-DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0466 0.1186 0.2161 0.2161 0.1992  0.0508
            P-value 0.1250 0.0040 0.0000 0.0000 0.0000  0.0010
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [368]:
# Event-DTW (2020) bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="event_dtw", parameter=1)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Event-DTW vs WisePanda:")
print(table)

Event-DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0805 0.2161 0.3559 0.4661 0.5847  0.4915
            P-value 0.0030 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.3 Modified COW vs WisePanda

In [376]:
# Modified COW (2022)
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="modified_cow", parameter=8)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Modified COW vs WisePanda:")
print(table)

Modified COW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0424 0.1059 0.1229 0.1271 0.0847  0.0212
            P-value 0.1770 0.0130 0.0060 0.0060 0.0040  0.0610
        Significant     No    Yes    Yes    Yes    Yes      No


In [395]:
# Modified COW (2022) bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="modified_cow", parameter=8)
table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Modified COW vs WisePanda:")
print(table)

Modified COW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1229 0.3305 0.3856 0.4746 0.6398  0.6059
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.4 Two-stage DTW vs WisePanda

In [377]:
# Two-stage DTW（2025）
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="two_stage_dtw", parameter=5)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Two-stage DTW vs WisePanda:")
print(table)

Two-stage DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0593 0.1483 0.1907 0.1441 0.1653  0.0466
            P-value 0.0520 0.0020 0.0000 0.0000 0.0000  0.0000
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [396]:
# Two-stage DTW（2025）bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="two_stage_dtw", parameter=5)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Two-stage DTW vs WisePanda:")
print(table)

Two-stage DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0932 0.2712 0.3432 0.4449 0.5847  0.5085
            P-value 0.0010 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.5 Fast Matching Method vs WisePanda

In [382]:
# Fast Matching Method
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="fast_matching", parameter=1)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("fast_matching vs WisePanda:")
print(table)

fast_matching vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0466 0.1992 0.2331 0.2542 0.2585  0.0551
            P-value 0.1350 0.0000 0.0000 0.0000 0.0000  0.0010
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [402]:
# Fast Matching Method
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="fast_matching", parameter=1)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("fast_matching vs WisePanda:")
print(table)

0.13559322033898305 0.08898305084745763
0.3855932203389831 0.1864406779661017
0.4957627118644068 0.2627118644067797
0.635593220338983 0.3813559322033898
0.9110169491525424 0.652542372881356
1.0 0.9449152542372882
fast_matching vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0466 0.1992 0.2331 0.2542 0.2585  0.0551
            P-value 0.1490 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [391]:
# Fast Matching Method bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="fast_matching", parameter=1)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("fast_matching vs WisePanda:")
print(table)

fast_matching vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0932 0.2797 0.3475 0.4703 0.6780  0.6186
            P-value 0.0010 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.6 DTW vs WisePanda

In [386]:
# DTW
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="dtw", parameter=1)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("DTW vs WisePanda:")
print(table)

DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0593 0.1610 0.1864 0.1398 0.1568  0.0508
            P-value 0.0500 0.0000 0.0000 0.0030 0.0000  0.0010
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [392]:
# DTW bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="dtw", parameter=1)

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("DTW vs WisePanda:")
print(table)

DTW vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0932 0.2839 0.3475 0.4492 0.5932  0.5212
            P-value 0.0010 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.1.7 Scale Invariant Signature vs WisePanda

In [383]:
# sis 
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method="sis", parameter='l2')

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Scale Invariant Signature  vs WisePanda:")
print(table)

Scale Invariant Signature  vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0297 0.1653 0.1864 0.1653 0.1822  0.0593
            P-value 0.3900 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant     No    Yes    Yes    Yes    Yes     Yes


In [393]:
# sis bamboo1350
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method="sis", parameter='l2')

valid_info = ['normal', 'wisepanda']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(2)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(2)+'.pth'
wisepanda_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Scale Invariant Signature  vs WisePanda:")
print(table)

Scale Invariant Signature  vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0847 0.2458 0.3347 0.4280 0.5932  0.5508
            P-value 0.0010 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


### 7.2 Generated model methods

#### 7.2.1 GAN vs WisePanda

In [371]:
# GAN
valid_info = ['normal', 'GAN']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("GAN vs WisePanda:")
print(table)

GAN vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0890 0.1822 0.2373 0.1737 0.1356  0.0424
            P-value 0.0010 0.0000 0.0000 0.0010 0.0000  0.0010
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


In [387]:
# GAN bamboo1350
valid_info = ['normal', 'GAN']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("GAN vs WisePanda:")
print(table)

GAN vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1229 0.3263 0.3941 0.4788 0.5890  0.5466
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.2.2 seriesGAN vs WisePanda

In [372]:
# seriesGAN
valid_info = ['normal', 'seriesGAN']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("seriesGAN vs WisePanda:")
print(table)

seriesGAN vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0636 0.1949 0.2458 0.2034 0.0932  0.0339
            P-value 0.0270 0.0000 0.0000 0.0000 0.0010  0.0070
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


In [388]:
# seriesGAN bamboo1350
valid_info = ['normal', 'seriesGAN']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("seriesGAN vs WisePanda:")
print(table)

seriesGAN vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1271 0.3263 0.3856 0.4237 0.5254  0.4492
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.2.3 Diffusion-TS  vs WisePanda

In [385]:
# Diffusion-TS
valid_info = ['normal', 'Diffusion-TS']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Diffusion-TS vs WisePanda:")
print(table)

Diffusion-TS vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1186 0.2500 0.2542 0.2076 0.1483  0.0212
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0450
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


In [389]:
# Diffusion-TS bamboo1350
valid_info = ['normal', 'Diffusion-TS']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Diffusion-TS vs WisePanda:")
print(table)

Diffusion-TS vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1314 0.3771 0.4746 0.5763 0.7712  0.7669
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


#### 7.2.4 Diffusion vs WisePanda

In [374]:
# Diffusion
valid_info = ['normal', 'Diffusion']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 0, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Diffusion vs WisePanda:")
print(table)

Diffusion vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.0847 0.2542 0.3263 0.3517 0.1822  0.0339
            P-value 0.0010 0.0000 0.0000 0.0000 0.0000  0.0040
        Significant    Yes    Yes    Yes    Yes    Yes     Yes


In [390]:
# Diffusion bamboo1350
valid_info = ['normal', 'Diffusion']
f_pth = 'models/'+valid_info[0]+'/f_model_'+valid_info[1]+str(1)+'.pth'
c_pth = 'models/'+valid_info[0]+'/c_model_'+valid_info[1]+str(1)+'.pth'
current_dis_map, result_list = get_topk_accuracy_with_extend(extend_number = 1114, method=valid_info[1], f_model_pth=f_pth, c_model_pth=c_pth)

table = run_permutation_test(current_dis_map, wisepanda_dis_map)
print("Diffusion vs WisePanda:")
print(table)

Diffusion vs WisePanda:
              Top-k  Top-1  Top-5 Top-10 Top-20 Top-50 Top-100
Observed Difference 0.1186 0.3263 0.3814 0.4831 0.6356  0.6017
            P-value 0.0000 0.0000 0.0000 0.0000 0.0000  0.0000
        Significant    Yes    Yes    Yes    Yes    Yes     Yes
